# Análisis de Selección de Features — ¿Cuántas variables necesita el modelo?
## Proyecto: Productividad Asesores de Negocios

---

### Objetivo

Determinar cuántas features necesita el modelo LightGBM para mantener
un macro F1 aceptable (caída < 2pp respecto al modelo completo con 22 features).

Esto define cuántas variables incluir en el dashboard de perfil de asesor
y cuántas sliders tiene sentido mostrar al usuario.


## 1. Configuración y carga de datos

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer

warnings.filterwarnings('ignore')

PROCESSED = Path('../data/processed')
FIG_DIR   = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Cargar datos procesados
data          = np.load(PROCESSED / 'features_processed.npz', allow_pickle=True)
X_train       = data['X_train']
X_test        = data['X_test']
y_train       = data['y_train']
y_test        = data['y_test']
feature_names = list(data['feature_names'])

# Limpiar prefijos del ColumnTransformer
feature_names_clean = [
    n.replace('num__', '').replace('cat__', '')
    for n in feature_names
]

# Cargar modelo y encoders
modelo        = joblib.load(PROCESSED / 'modelo_final.joblib')
label_encoder = joblib.load(PROCESSED / 'label_encoder.joblib')
CLASS_NAMES   = list(label_encoder.classes_)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Clases  : {CLASS_NAMES}')
print(f'Features: {len(feature_names_clean)}')


X_train : (1603, 22)
X_test  : (401, 22)
Clases  : ['Q1_BAJO', 'Q2_MEDIO_BAJO', 'Q3_MEDIO_ALTO', 'Q4_ALTO']
Features: 22


## 2. Importancia nativa de LightGBM

In [2]:
# Importancia nativa de LightGBM (número de splits por feature)
# Distinta a SHAP pero útil para selección de features
importancia_lgbm = pd.DataFrame({
    'feature'   : feature_names_clean,
    'importance': modelo.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('=== IMPORTANCIA NATIVA LIGHTGBM (orden para selección) ===')
display(importancia_lgbm)

# Visualización
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(importancia_lgbm['feature'][::-1],
        importancia_lgbm['importance'][::-1],
        color='#4C72B0')
ax.set_xlabel('Importancia (número de splits)', fontsize=11)
ax.set_title('Importancia Nativa LightGBM — Todas las features', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '11_feature_importance_nativa.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print('Figura guardada: 11_feature_importance_nativa.png')


=== IMPORTANCIA NATIVA LIGHTGBM (orden para selección) ===


,feature,importance
0,INCREMENTO_CARTERA,485
1,TASA_PROM,431
2,CLIENTES_PRESTAMO,429
3,CLIENTES,416
4,N_SEMANAS_OBS,316
5,CLIENTES_NUEVOS,306
6,PRESTAMO,302
7,DESEMBOLSO_CLIENTES_NUEVOS,247
8,EDAD,228
9,GRUPOS,223


Figura guardada: 11_feature_importance_nativa.png


## 3. Evaluación con Top N features

Entrenamos el modelo con los mismos hiperparámetros del Sprint 5
pero usando solo las top N features según importancia nativa.

**Criterio de decisión:** elegir el N más pequeño con caída < 2pp vs el modelo completo.


In [3]:
# Hiperparámetros del mejor modelo del Sprint 5 (hardcodeados para reproducibilidad)
BEST_PARAMS = {
    'n_estimators'      : 175,
    'learning_rate'     : 0.13625252988274353,
    'num_leaves'        : 26,
    'max_depth'         : 3,
    'min_child_samples' : 32,
    'subsample'         : 0.8546743841357609,
    'colsample_bytree'  : 0.9581541304120595,
    'reg_alpha'         : 0.0011949356785869657,
    'reg_lambda'        : 0.047781950780058036,
}

# Scorer y CV
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average='macro')

F1_MODELO_COMPLETO = 0.6144  # Referencia del Sprint 5

resultados = []
print(f'Referencia — modelo completo (22 features): Test F1 = {F1_MODELO_COMPLETO}')
print(f'Umbral de caida aceptable: < 0.02 (2pp)')
print('─' * 70)

for n in [4, 6, 8, 10, 12, 15, 18, 22]:
    # Seleccionar indices de las top N features
    top_idx   = importancia_lgbm.index[:n].tolist()
    X_train_n = X_train[:, top_idx]
    X_test_n  = X_test[:, top_idx]

    # Entrenar modelo con mismos hiperparametros
    modelo_n = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=4,
        verbosity=-1,
        random_state=42,
        is_unbalance=True,
        **BEST_PARAMS
    )

    # Validacion cruzada
    cv_scores = cross_val_score(
        modelo_n, X_train_n, y_train,
        cv=cv, scoring=scorer, n_jobs=-1
    )

    # Evaluacion en test
    modelo_n.fit(X_train_n, y_train)
    y_pred_n = modelo_n.predict(X_test_n)
    test_f1  = f1_score(y_test, y_pred_n, average='macro')
    caida    = F1_MODELO_COMPLETO - test_f1

    # Nombres de las features seleccionadas
    feats_sel = importancia_lgbm['feature'].iloc[:n].tolist()

    resultados.append({
        'n_features'  : n,
        'cv_f1_mean'  : round(cv_scores.mean(), 4),
        'cv_f1_std'   : round(cv_scores.std(), 4),
        'test_f1'     : round(test_f1, 4),
        'caida_vs_22' : round(caida, 4),
        'aceptable'   : 'SI' if caida < 0.02 else 'NO',
        'features'    : feats_sel,
    })

    marca = '<-- UMBRAL OK' if caida < 0.02 else ''
    print(f'Top {n:2d} -> CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} | '
          f'Test F1: {test_f1:.4f} | Caida: {caida:+.4f} {marca}')

df_res = pd.DataFrame(resultados)


Referencia — modelo completo (22 features): Test F1 = 0.6144
Umbral de caida aceptable: < 0.02 (2pp)
──────────────────────────────────────────────────────────────────────
Top  4 -> CV F1: 0.5566 +/- 0.0087 | Test F1: 0.5630 | Caida: +0.0514 
Top  6 -> CV F1: 0.5887 +/- 0.0260 | Test F1: 0.5809 | Caida: +0.0335 
Top  8 -> CV F1: 0.5967 +/- 0.0158 | Test F1: 0.6126 | Caida: +0.0018 <-- UMBRAL OK
Top 10 -> CV F1: 0.5911 +/- 0.0317 | Test F1: 0.6111 | Caida: +0.0033 <-- UMBRAL OK
Top 12 -> CV F1: 0.5997 +/- 0.0145 | Test F1: 0.6258 | Caida: -0.0114 <-- UMBRAL OK
Top 15 -> CV F1: 0.6083 +/- 0.0226 | Test F1: 0.6147 | Caida: -0.0003 <-- UMBRAL OK
Top 18 -> CV F1: 0.6060 +/- 0.0222 | Test F1: 0.6085 | Caida: +0.0059 <-- UMBRAL OK
Top 22 -> CV F1: 0.6138 +/- 0.0229 | Test F1: 0.6144 | Caida: +0.0000 <-- UMBRAL OK


## 4. Tabla resumen y decisión

In [4]:
print('=== TABLA RESUMEN ===')
display(df_res[['n_features','cv_f1_mean','cv_f1_std','test_f1','caida_vs_22','aceptable']])

print('\n=== FEATURES POR GRUPO ===')
for _, row in df_res.iterrows():
    n     = int(row['n_features'])
    feats = row['features']
    marca = '<-- MINIMO ACEPTABLE' if row['aceptable'] == 'SI' and \
            (df_res[df_res['aceptable']=='SI']['n_features'].min() == n) else ''
    print(f'\nTop {n:2d} {marca}')
    for f in feats:
        print(f'  - {f}')

# Identificar el minimo N aceptable
n_min = df_res[df_res['aceptable'] == 'SI']['n_features'].min()
row_min = df_res[df_res['n_features'] == n_min].iloc[0]

print(f'\n' + '='*60)
print(f'DECISION: usar Top {n_min} features')
print(f'  Test F1   : {row_min["test_f1"]}')
print(f'  Caida     : {row_min["caida_vs_22"]:+.4f} ({row_min["caida_vs_22"]*100:.1f}pp)')
print(f'  Features  : {row_min["features"]}')
print('='*60)


=== TABLA RESUMEN ===


,n_features,cv_f1_mean,cv_f1_std,test_f1,caida_vs_22,aceptable
0,4,0.5566,0.0087,0.5630,0.0514,NO
1,6,0.5887,0.0260,0.5809,0.0335,NO
2,8,0.5967,0.0158,0.6126,0.0018,SI
3,10,0.5911,0.0317,0.6111,0.0033,SI
4,12,0.5997,0.0145,0.6258,-0.0114,SI
5,15,0.6083,0.0226,0.6147,-0.0003,SI
6,18,0.6060,0.0222,0.6085,0.0059,SI
7,22,0.6138,0.0229,0.6144,0.0000,SI



=== FEATURES POR GRUPO ===

Top  4 
  - INCREMENTO_CARTERA
  - TASA_PROM
  - CLIENTES_PRESTAMO
  - CLIENTES

Top  6 
  - INCREMENTO_CARTERA
  - TASA_PROM
  - CLIENTES_PRESTAMO
  - CLIENTES
  - N_SEMANAS_OBS
  - CLIENTES_NUEVOS

Top  8 <-- MINIMO ACEPTABLE
  - INCREMENTO_CARTERA
  - TASA_PROM
  - CLIENTES_PRESTAMO
  - CLIENTES
  - N_SEMANAS_OBS
  - CLIENTES_NUEVOS
  - PRESTAMO
  - DESEMBOLSO_CLIENTES_NUEVOS

Top 10 
  - INCREMENTO_CARTERA
  - TASA_PROM
  - CLIENTES_PRESTAMO
  - CLIENTES
  - N_SEMANAS_OBS
  - CLIENTES_NUEVOS
  - PRESTAMO
  - DESEMBOLSO_CLIENTES_NUEVOS
  - EDAD
  - GRUPOS

Top 12 
  - INCREMENTO_CARTERA
  - TASA_PROM
  - CLIENTES_PRESTAMO
  - CLIENTES
  - N_SEMANAS_OBS
  - CLIENTES_NUEVOS
  - PRESTAMO
  - DESEMBOLSO_CLIENTES_NUEVOS
  - EDAD
  - GRUPOS
  - REGION
  - MOTIVO_BAJA

Top 15 
  - INCREMENTO_CARTERA
  - TASA_PROM
  - CLIENTES_PRESTAMO
  - CLIENTES
  - N_SEMANAS_OBS
  - CLIENTES_NUEVOS
  - PRESTAMO
  - DESEMBOLSO_CLIENTES_NUEVOS
  - EDAD
  - GRUPOS
  - REGION
  

## 5. Visualización — curva F1 vs número de features

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_res['n_features'], df_res['test_f1'],
        marker='o', linewidth=2, color='#4C72B0', label='Test F1')
ax.plot(df_res['n_features'], df_res['cv_f1_mean'],
        marker='s', linewidth=2, linestyle='--', color='#E8694A', label='CV F1 (media)')

# Linea de referencia: modelo completo
ax.axhline(y=F1_MODELO_COMPLETO, color='gray', linestyle=':', linewidth=1.5,
           label=f'Modelo completo ({F1_MODELO_COMPLETO})')

# Zona de caida aceptable
ax.axhspan(F1_MODELO_COMPLETO - 0.02, F1_MODELO_COMPLETO,
           alpha=0.1, color='green', label='Zona aceptable (caida < 2pp)')

# Marcar el N minimo aceptable
ax.axvline(x=n_min, color='green', linestyle='--', linewidth=1.5,
           label=f'Minimo aceptable: Top {n_min}')

ax.set_xlabel('Número de features', fontsize=11)
ax.set_ylabel('Macro F1', fontsize=11)
ax.set_title('Macro F1 vs Número de Features — Curva de selección', fontsize=12)
ax.set_xticks(df_res['n_features'].tolist())
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '12_feature_selection_curve.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print('Figura guardada: 12_feature_selection_curve.png')


Figura guardada: 12_feature_selection_curve.png


## 6. Umbrales en unidades reales de negocio

In [6]:
# Recuperar los parametros del scaler para traducir valores escalados a reales
preprocessor = joblib.load(PROCESSED / 'preprocessor.joblib')
scaler = preprocessor.named_transformers_['num']['scaler']

NUM_COLS = [
    'GRUPOS', 'CLIENTES', 'PRESTAMO', 'CLIENTES_PRESTAMO',
    'TASA_PROM', 'INCREMENTO_CARTERA', 'CLIENTES_NUEVOS',
    'DESEMBOLSO_CLIENTES_NUEVOS', 'N_SEMANAS_OBS', 'EDAD',
]

print('=== ESTADISTICOS DEL SCALER — UNIDADES REALES ===')
print(f'{"Feature":<30} {"Media":>12} {"Std":>12} {"Min aprox":>12} {"Max aprox":>12}')
print('─' * 70)
for i, col in enumerate(NUM_COLS):
    media = scaler.mean_[i]
    std   = scaler.scale_[i]
    min_aprox = media - 2 * std
    max_aprox = media + 2 * std
    print(f'{col:<30} {media:>12.2f} {std:>12.2f} {min_aprox:>12.2f} {max_aprox:>12.2f}')

print('\nNota: Min/Max aprox = media +/- 2 desviaciones estandar (cubre ~95% del rango)')
print('Estos valores definen los rangos de los sliders del dashboard.')


=== ESTADISTICOS DEL SCALER — UNIDADES REALES ===
Feature                               Media          Std    Min aprox    Max aprox
──────────────────────────────────────────────────────────────────────
GRUPOS                                 1.02         0.06         0.90         1.14
CLIENTES                               9.75         1.04         7.66        11.83
PRESTAMO                            9247.54     16342.49    -23437.44     41932.52
CLIENTES_PRESTAMO                      0.86         1.16        -1.47         3.18
TASA_PROM                            218.71        43.79       131.12       306.30
INCREMENTO_CARTERA                  5157.85     15872.13    -26586.42     36902.12
CLIENTES_NUEVOS                        0.29         0.67        -1.05         1.64
DESEMBOLSO_CLIENTES_NUEVOS          2105.08     12081.20    -22057.33     26267.49
N_SEMANAS_OBS                         33.82        32.00       -30.18        97.83
EDAD                                  30.88      